# 03 — BERTopic

BERTopic owns its own embedding (MiniLM) and clustering (UMAP + HDBSCAN
internally), so it's kept separate from notebook 02 rather than reusing
its cached embeddings.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import json

import numpy as np
import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP

from utils import config
from utils.data import stratified_sample
from utils.metrics import evaluate_unsupervised

In [2]:
train_clean = pd.read_parquet(config.PROCESSED_DIR / "train_clean.parquet")
train_sample = stratified_sample(train_clean, config.SAMPLE_SIZE, seed=config.SEED)
texts = train_sample["text"].tolist()
true_labels = train_sample["label"].to_numpy()

In [3]:
umap_model = UMAP(n_neighbors=15, n_components=5, metric="cosine", random_state=config.SEED)
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
topic_model = BERTopic(embedding_model=sentence_model, umap_model=umap_model,
                        nr_topics=config.NUM_CLASSES, calculate_probabilities=False)

topics, _ = topic_model.fit_transform(texts)
print(topic_model.get_topic_info())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

   Topic  Count             Name  \
0     -1   2849  -1_the_to_of_in   
1      0   2392   0_the_to_of_in   
2      1   1510  1_the_to_in_and   
3      2   1249   2_the_in_to_of   

                                    Representation  \
0  [the, to, of, in, and, for, on, 39, that, with]   
1   [the, to, of, in, and, on, for, its, 39, that]   
2    [the, to, in, and, of, 39, for, on, at, with]   
3  [the, in, to, of, on, and, for, said, iraq, 39]   

                                 Representative_Docs  
0  [Microsoft Looks to Expand Windows at Home (AP...  
1  [Red hot Google shares cooling  Google Inc. ma...  
2  [Radcliffe Decides to Race in NYC Marathon (AP...  
3  [Sudanese government and rebels agree to end s...  


In [4]:
topics_arr = np.array(topics)
topic_embeddings = sentence_model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
bertopic_metrics = evaluate_unsupervised(true_labels, topics_arr, topic_embeddings)
print(bertopic_metrics)

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(config.RESULTS_DIR / "metrics_bertopic.json", "w") as f:
    json.dump(bertopic_metrics, f, indent=2)
print("Saved BERTopic metrics.")

Batches:   0%|          | 0/250 [00:00<?, ?it/s]

{'ACC (Hungarian)': np.float64(0.7165598912832459), 'NMI': 0.6605692501355472, 'ARI': 0.6126857294510963, 'FMI': 0.7391833003955273, 'Homogeneity': 0.5832207743444828, 'Completeness': 0.7615710737154081, 'V-Measure': 0.6605692501355473, 'Silhouette Score': 0.07383336871862411, 'Davies-Bouldin': 4.8054618184335185, 'Coverage': np.float64(0.643875)}
Saved BERTopic metrics.
